<a href="https://colab.research.google.com/github/Tharun-sk07/DAA/blob/main/extra_problem_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Matrix Chain Multiplication: Memoized (Top-Down) vs. Bottom-Up DP

This notebook will demonstrate two approaches to solve the Matrix Chain Multiplication problem: a memoized (top-down) recursive solution and a bottom-up dynamic programming solution. We will then compare their results for a chain of 6 matrices.

### 1. Memoized (Top-Down) Approach

This approach uses recursion with caching (memoization) to avoid recomputing the same subproblems. The base case is when `i == j`, meaning a single matrix, which requires 0 multiplications. Otherwise, it explores all possible split points `k` and takes the minimum cost, storing the result in a `memo` table.

In [1]:
def matrix_chain_order_memoized(p, i, j, memo):
    """
    Computes the minimum number of scalar multiplications needed to multiply
    the matrix chain p[i..j] using memoization.

    Args:
        p (list): List of matrix dimensions, where p[k] is the number of columns
                  for matrix A_k and p[k-1] is the number of rows for A_k.
                  So, matrix A_k has dimensions p[k-1] x p[k].
        i (int): Starting index of the matrix chain (0-indexed).
        j (int): Ending index of the matrix chain (0-indexed).
        memo (dict): Dictionary to store computed results for (i, j).

    Returns:
        int: Minimum number of scalar multiplications.
    """
    if i == j:
        return 0

    if (i, j) in memo:
        return memo[(i, j)]

    min_cost = float('inf')

    # Iterate through all possible split points k
    for k in range(i, j):
        cost = (matrix_chain_order_memoized(p, i, k, memo) +
                matrix_chain_order_memoized(p, k + 1, j, memo) +
                p[i - 1] * p[k] * p[j])
        min_cost = min(min_cost, cost)

    memo[(i, j)] = min_cost
    return min_cost

def mcm_memoized_wrapper(p):
    n = len(p) - 1 # Number of matrices
    memo = {}
    # The function uses 1-based indexing for matrices for easier formula mapping
    # so we call it with (1, n)
    return matrix_chain_order_memoized(p, 1, n, memo)


### 2. Bottom-Up Dynamic Programming Approach

This approach builds up the solution from the smallest subproblems. It uses a 2D `dp` table where `dp[i][j]` stores the minimum cost to multiply matrices from `i` to `j`. The table is filled diagonally, considering chains of increasing length.

In [3]:
def matrix_chain_order_bottom_up(p):
    """
    Computes the minimum number of scalar multiplications needed to multiply
    the matrix chain using bottom-up dynamic programming.

    Args:
        p (list): List of matrix dimensions, where p[k] is the number of columns
                  for matrix A_k and p[k-1] is the number of rows for A_k.
                  So, matrix A_k has dimensions p[k-1] x p[k].

    Returns:
        int: Minimum number of scalar multiplications.
    """
    n = len(p) - 1  # Number of matrices

    # dp[i][j] stores the minimum number of scalar multiplications
    # needed to compute the matrix product A_i * A_{i+1} * ... * A_j.
    # Using 1-based indexing for matrices, so array size is (n+1) x (n+1).
    dp = [[0 for _ in range(n + 1)] for _ in range(n + 1)]

    # Base cases: dp[i][i] = 0 for all i (single matrix, 0 multiplications)
    # This is already handled by initializing with 0s

    # L is the chain length (number of matrices in the subproblem)
    for L in range(2, n + 1):  # L goes from 2 to n
        for i in range(1, n - L + 2):  # i goes from 1 to n - L + 1
            j = i + L - 1  # j is the end index of the chain
            dp[i][j] = float('inf')

            # Iterate through all possible split points k
            for k in range(i, j):
                # Cost of multiplying A_i..A_k + A_{k+1}..A_j +
                # Cost of multiplying the resulting two matrices
                cost = dp[i][k] + dp[k + 1][j] + p[i - 1] * p[k] * p[j]
                dp[i][j] = min(dp[i][j], cost)

    return dp[1][n]


### 3. Comparison and Verification for 6 Matrices

We will now define a chain of 6 matrices and compare the results from both the memoized and bottom-up DP approaches. The `p` array will contain 7 dimensions for 6 matrices (e.g., `p = [d0, d1, d2, d3, d4, d5, d6]` for matrices A1(d0xd1), A2(d1xd2), ..., A6(d5xd6)).

In [4]:
# Define dimensions for 6 matrices (e.g., A1(30x35), A2(35x15), A3(15x5), A4(5x10), A5(10x20), A6(20x25))
# The array p has n+1 elements for n matrices.
# p[0]x p[1], p[1]x p[2], ..., p[n-1]x p[n]
# Here, we have 6 matrices, so p has 7 elements.
# Example: p = [30, 35, 15, 5, 10, 20, 25]
matrix_dimensions = [30, 35, 15, 5, 10, 20, 25]

print(f"Matrix dimensions (p array): {matrix_dimensions}")

# --- Memoized (Top-Down) Approach ---
memoized_cost = mcm_memoized_wrapper(matrix_dimensions)
print(f"\nMinimum cost using Memoized (Top-Down) DP: {memoized_cost}")

# --- Bottom-Up Dynamic Programming Approach ---
bottom_up_cost = matrix_chain_order_bottom_up(matrix_dimensions)
print(f"Minimum cost using Bottom-Up DP: {bottom_up_cost}")

# --- Verification ---
if memoized_cost == bottom_up_cost:
    print("\nVerification Successful: Both approaches yield the same minimum cost!")
else:
    print("\nVerification Failed: The costs from the two approaches do not match.")

assert memoized_cost == bottom_up_cost, "Costs do not match!"


Matrix dimensions (p array): [30, 35, 15, 5, 10, 20, 25]

Minimum cost using Memoized (Top-Down) DP: 15125
Minimum cost using Bottom-Up DP: 15125

Verification Successful: Both approaches yield the same minimum cost!
